# Configuração do Ambiente

In [0]:
%pip install openpyxl

In [0]:
dbutils.library.restartPython()

In [0]:

########################################################
#### LIBS
########################################################
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    BooleanType, DoubleType, TimestampType, LongType
)
from pyspark.sql.window import Window
import pyspark
#from pyspark.sql.functions import col
import pandas as pd
import numpy as np
import requests
import json
import os
import hashlib
import json as _json
from datetime import datetime, timezone

# Leitura dos dados

In [0]:
# evita o erro de tipos misturados
ibge = pd.read_excel("/Volumes/workspace/default/inep_avaliacao_alfabetizacao/bronze/batch/ibge_agregados_municipios_basico_BR.xlsx", dtype=str) 
ibge = ibge[["CD_MUN", "NM_MUN", "CD_REGIAO", "NM_REGIAO", "CD_UF", "NM_UF", "AREA_KM2", "v0001"]]# seleção no pandas
ibge = ibge.where(ibge.notna(), None) #NaN -> None

##conversão em DataFrame Spark
ibge = spark.createDataFrame(ibge)

In [0]:
print("Informação sobre o tipo das variáveis e se permite valores nulos:")
ibge.printSchema()
print("prévia da tabela:")
ibge.show(5, truncate=False)

# Tratamento dos dados

In [0]:
##
ibge = ibge.withColumnRenamed("v0001", "POPULACAO")

#Conversão do tipo da coluna
ibge = (
    ibge
    .withColumn("AREA_KM2", F.col("AREA_KM2").cast("double"))  
    .withColumn("POPULACAO",    F.col("POPULACAO").cast("long"))     
)

ibge.show(5, truncate=False)

# Exportação dos dados

In [0]:

#
file_path =  "/Volumes/workspace/default/inep_avaliacao_alfabetizacao/silver/ibge_censo_municipios_basico"
#
ibge.write.mode("overwrite").parquet(file_path)